# Tennis Vision — Colab Runner

---
## Sections
- **[A] Setup** — clone, install, models
- **[B] Process Video** — analyze your YouTube Shorts
- **[C] Fine-tune Ball Detector** — YOLOv8 on tennis ball dataset
- **[D] Fine-tune Court Detector** — better in/out & speed
- **[E] Save/Load Models** — persist to Google Drive
---
### First time:
```
A1 → A2 → A3 → C1 → C2 → D1 → D2 → E1 → B1 → B2
```
### Next sessions (already trained):
```
A1 → A2 → A3 → E2 (load from Drive) → B1
```

> **Before running C1:** Add your Roboflow key to Colab Secrets
> Left sidebar → 🔑 key icon → `+ Add new secret`
> Name: `ROBOFLOW_API_KEY`  |  Value: your key

## Section A — Setup

In [ ]:
# ── A1: Clone / pull latest ──────────────────────────────────────────────────
import os

if not os.path.exists('/content/tennis_vision'):
    !git clone https://github.com/bozanctn/tennis_vision.git /content/tennis_vision
    print('Cloned.')
else:
    !git -C /content/tennis_vision pull origin main
    print('Pulled latest.')

%cd /content/tennis_vision

In [ ]:
# ── A2: Install dependencies ─────────────────────────────────────────────────
!pip install -q -r requirements.txt
!pip install -q roboflow
print('Done.')

In [ ]:
# ── A3: Download base YOLOv8 weights ─────────────────────────────────────────
!python scripts/download_models.py

## Section B — Process Your Video

In [ ]:
# ── B1: Download + process YouTube Shorts ────────────────────────────────────
YOUTUBE_URL = 'https://www.youtube.com/shorts/08-cKvAJMVo'
OUTPUT_FILE = '/content/tennis_vision/result.mp4'

!python scripts/process_video.py \
    --input "{YOUTUBE_URL}" \
    --output "{OUTPUT_FILE}" \
    --device cuda

print(f'Done → {OUTPUT_FILE}')

In [ ]:
# ── B2: Preview inside Colab ──────────────────────────────────────────────────
from IPython.display import HTML
from base64 import b64encode
import os

f = '/content/tennis_vision/result.mp4'
p = f.replace('.mp4', '_preview.mp4')
!ffmpeg -y -i "{f}" -vcodec libx264 -acodec aac "{p}" -loglevel quiet

with open(p, 'rb') as fh:
    b64 = b64encode(fh.read()).decode()
display(HTML(f'<video width="720" controls><source src="data:video/mp4;base64,{b64}" type="video/mp4"></video>'))

In [ ]:
# ── B3: Download to your computer ────────────────────────────────────────────
from google.colab import files
files.download('/content/tennis_vision/result.mp4')

## Section C — Fine-tune YOLOv8 Ball Detector

Trains YOLOv8n on a real tennis ball dataset (~1700 images) so it can reliably
detect the ball at different distances, speeds, and lighting conditions.

**Setup Roboflow API key (one time only):**
1. Left sidebar → 🔑 icon (Secrets)
2. `+ Add new secret`
3. Name: `ROBOFLOW_API_KEY` — Value: `IqeiVYOZpAA8YlTybFzZ`
4. Toggle **Notebook access** ON
5. Done — never needs to be done again

In [ ]:
# ── C1: Load API key from Colab Secrets (safe — never stored in code) ─────────
from google.colab import userdata
ROBOFLOW_API_KEY = userdata.get('ROBOFLOW_API_KEY')
print('API key loaded.' if ROBOFLOW_API_KEY else 'ERROR: key not found — check Secrets setup above.')

In [ ]:
# ── C2: Download tennis ball dataset (auto, ~1700 images) ─────────────────────
# Tries multiple known public datasets, uses first one that works
from roboflow import Roboflow
import os

os.makedirs('/content/tennis_vision/data', exist_ok=True)
os.chdir('/content/tennis_vision/data')

rf = Roboflow(api_key=ROBOFLOW_API_KEY)

BALL_DATASET_PATH = None
candidates = [
    ('roboflow-jvuqo',  'tennis-ball-detection-tracknet', 4),
    ('tennis-nimqk',    'tennis-ball-detection-dkqrp',    1),
    ('david-mchale',    'tennis-ball',                    1),
]

for workspace, project_name, version in candidates:
    try:
        print(f'Trying {workspace}/{project_name} v{version}...')
        project = rf.workspace(workspace).project(project_name)
        dataset = project.version(version).download('yolov8')
        BALL_DATASET_PATH = dataset.location
        print(f'Downloaded to: {BALL_DATASET_PATH}')
        break
    except Exception as e:
        print(f'  Failed: {e}')

os.chdir('/content/tennis_vision')

if BALL_DATASET_PATH is None:
    print('\nAll datasets failed. Go to https://universe.roboflow.com and search')
    print('"tennis ball" → pick any dataset → Export → YOLOv8 → copy the code snippet')
else:
    print(f'\nReady to train!')

In [ ]:
# ── C3: Train ball detector (~15 min on T4) ───────────────────────────────────
!yolo train \
  model=yolov8n.pt \
  data="{BALL_DATASET_PATH}/data.yaml" \
  epochs=50 \
  imgsz=640 \
  batch=16 \
  project=/content/tennis_vision/models \
  name=ball_detector \
  device=0 \
  exist_ok=True

In [ ]:
# ── C4: Validate + update config ─────────────────────────────────────────────
from ultralytics import YOLO
from IPython.display import Image, display
import os, yaml

best = '/content/tennis_vision/models/ball_detector/weights/best.pt'
assert os.path.exists(best), 'Training failed — check C3 output'

model = YOLO(best)
metrics = model.val(data=f'{BALL_DATASET_PATH}/data.yaml', device=0)
print(f'\nmAP@50    : {metrics.box.map50:.3f}  (aim > 0.70)')
print(f'Precision : {metrics.box.mp:.3f}')
print(f'Recall    : {metrics.box.mr:.3f}')

img_path = '/content/tennis_vision/models/ball_detector/results.png'
if os.path.exists(img_path):
    display(Image(img_path, width=900))

# Update config
with open('/content/tennis_vision/config/config.yaml') as f:
    cfg = yaml.safe_load(f)
cfg['models']['ball_model_path'] = 'models/ball_detector/weights/best.pt'
with open('/content/tennis_vision/config/config.yaml', 'w') as f:
    yaml.dump(cfg, f, default_flow_style=False)
print('\nConfig updated — run B1 now!')

## Section D — Fine-tune Court Detector

Trains a court line detector → more accurate homography → better in/out calls and speed.

In [ ]:
# ── D1: Download court dataset (auto) ────────────────────────────────────────
from roboflow import Roboflow
import os

os.makedirs('/content/tennis_vision/data', exist_ok=True)
os.chdir('/content/tennis_vision/data')

rf = Roboflow(api_key=ROBOFLOW_API_KEY)

COURT_DATASET_PATH = None
candidates = [
    ('tennis-court-zqmfm', 'tennis-court-detection-uq5vc', 1),
    ('sport-field-jvjjf',  'tennis-court',                 1),
    ('roboflow-jvuqo',     'tennis-court-detection',        1),
]

for workspace, project_name, version in candidates:
    try:
        print(f'Trying {workspace}/{project_name} v{version}...')
        project = rf.workspace(workspace).project(project_name)
        dataset = project.version(version).download('yolov8')
        COURT_DATASET_PATH = dataset.location
        print(f'Downloaded to: {COURT_DATASET_PATH}')
        break
    except Exception as e:
        print(f'  Failed: {e}')

os.chdir('/content/tennis_vision')

if COURT_DATASET_PATH is None:
    print('\nAll datasets failed — search "tennis court" on universe.roboflow.com')

In [ ]:
# ── D2: Train court detector (~25 min on T4) ──────────────────────────────────
# Higher resolution (1280) because court lines are thin
!yolo train \
  model=yolov8n.pt \
  data="{COURT_DATASET_PATH}/data.yaml" \
  epochs=80 \
  imgsz=1280 \
  batch=8 \
  project=/content/tennis_vision/models \
  name=court_detector \
  device=0 \
  exist_ok=True

In [ ]:
# ── D3: Update config ─────────────────────────────────────────────────────────
import yaml

with open('/content/tennis_vision/config/config.yaml') as f:
    cfg = yaml.safe_load(f)
cfg['models']['court_model_path'] = 'models/court_detector/weights/best.pt'
with open('/content/tennis_vision/config/config.yaml', 'w') as f:
    yaml.dump(cfg, f, default_flow_style=False)
print('Court model configured — run B1!')

## Section E — Save / Load Models (Google Drive)
Save once after training → load in 5 seconds next session instead of retraining.

In [ ]:
# ── E1: Save all models to Drive ─────────────────────────────────────────────
from google.colab import drive
import shutil, os

drive.mount('/content/drive')
drive_dir = '/content/drive/MyDrive/tennis_vision_models'
os.makedirs(drive_dir, exist_ok=True)

to_save = {
    '/content/tennis_vision/models/ball_detector/weights/best.pt':  'ball_detector.pt',
    '/content/tennis_vision/models/court_detector/weights/best.pt': 'court_detector.pt',
}
for src, name in to_save.items():
    if os.path.exists(src):
        shutil.copy(src, f'{drive_dir}/{name}')
        print(f'Saved {name}')
    else:
        print(f'Skipped {name} — not found')

print(f'\nSaved to: {drive_dir}')

In [ ]:
# ── E2: Load models from Drive (use at start of every new session) ────────────
from google.colab import drive
import shutil, os, yaml

drive.mount('/content/drive')
drive_dir = '/content/drive/MyDrive/tennis_vision_models'
local_dir = '/content/tennis_vision/models'
os.makedirs(local_dir, exist_ok=True)

for name in ['ball_detector.pt', 'court_detector.pt']:
    src = f'{drive_dir}/{name}'
    if os.path.exists(src):
        shutil.copy(src, f'{local_dir}/{name}')
        print(f'Loaded {name}')
    else:
        print(f'Not found: {name} — run Section C/D first')

# Update config
with open('/content/tennis_vision/config/config.yaml') as f:
    cfg = yaml.safe_load(f)
if os.path.exists(f'{local_dir}/ball_detector.pt'):
    cfg['models']['ball_model_path']  = 'models/ball_detector.pt'
if os.path.exists(f'{local_dir}/court_detector.pt'):
    cfg['models']['court_model_path'] = 'models/court_detector.pt'
with open('/content/tennis_vision/config/config.yaml', 'w') as f:
    yaml.dump(cfg, f, default_flow_style=False)

print('\nReady — run B1!')